# 02.04 PyPTO API 与计算图索引

本节作为本章的收束章节，集中整理 PyPTO API、Operation 和计算图层次。上一节已经说明 Host/Kernel 分工和 MPMD 的直观含义；本节进一步回答：常见 API 分别描述什么，Tensor 表达如何对应到 Operation，框架为什么需要多层计算图。

本节的定位是“索引”和“归纳”，不是重新讲一遍 Hello World。后续进入逐元素算子、矩阵乘法、规约和 Softmax 时，可以用这里的分类方式快速判断一行代码处在 PyPTO 开发链路的哪个位置。


## 1. 最小依赖检查

本节只需要导入 `torch` 和 `pypto`。代码单元用于确认运行时依赖，不依赖任何源码目录或文档目录。


In [ ]:
import torch
import pypto

print("torch:", torch.__version__)
print("pypto:", pypto.__file__)


输出中的 `torch` 版本表示 Host 侧张量库可用，`pypto` 路径表示 PyPTO 包可以被当前 Python 环境导入。本节后续内容以概念和代码结构为主，不需要执行 NPU kernel。


## 2. PyPTO API 的系统分类

PyPTO API 可以先按“在开发链路中解决什么问题”分类。这个分类比单纯记 API 名称更重要，因为同一类 API 往往会在不同算子中反复出现。

| 类别 | 典型 API 或写法 | 解决的问题 | 常见位置 |
| --- | --- | --- | --- |
| JIT 与运行配置 | `pypto.frontend.jit`、`pypto.RunMode.NPU`、`pypto.RunMode.SIM` | 标记 kernel，指定运行模式 | Kernel 定义处 |
| Tensor 参数描述 | `pypto.Tensor([...], pypto.DT_FP32)` | 描述 kernel 输入输出的 shape 特征和 dtype | Kernel 函数签名 |
| 数据类型 | `pypto.DT_FP32`、`pypto.DT_FP16`、`pypto.DT_BF16` | 表示 Tensor 元素类型 | 参数描述、算子输出 dtype |
| Tile 配置 | `pypto.set_vec_tile_shapes`、`pypto.set_cube_tile_shapes` | 描述向量类或矩阵类计算的分块执行方式 | Kernel 函数内部 |
| Tensor 运算 | `x + y`、`pypto.add`、`pypto.matmul`、`pypto.sum`、`pypto.exp` | 形成计算图中的 Operation 节点 | Kernel 函数内部 |
| 输出写回 | `out[:] = ...`、`out.move(...)` | 把计算结果写入调用者提供的输出 Tensor | Kernel 函数内部 |
| 控制流 | `pypto.loop`、`pypto.cond` | 描述需要进入计算图的循环或条件逻辑 | 复杂 Kernel 内部 |
| 配置与调试 | `set_*_options`、`get_*_options` | 控制编译、验证、调试等行为 | 工程化调优阶段 |

这个表格中的 API 共同服务于一个目标：把 Tensor 级计算清楚地描述给 PyPTO，让框架能够记录依赖、生成图结构并组织执行。


## 3. 从 Tensor 表达到 Operation

在 PyPTO kernel 内部，Tensor 运算会形成计算图中的 Operation。以加法为例：

```python
out[:] = x + y
```

这行代码可以拆成两个层次理解：

| 片段 | 图中的含义 |
| --- | --- |
| `x + y` | 形成逐元素加法 Operation |
| `out[:] = ...` | 将 Operation 结果连接到输出 Tensor |

如果使用显式 API，含义接近：

```python
tmp = pypto.add(x, y)
out.move(tmp)
```

语法糖和显式 API 的共同点是：Tensor 运算描述“算什么”，输出写回描述“结果放到哪里”。在矩阵乘法、规约、Softmax 等算子中，图里的 Operation 会更多，但判断方式相同。


## 4. 计算图的四个层次

PyPTO 的计算图可以先分成四个层次理解。层次越靠前，越接近 Python 中写出的 Tensor 表达；层次越靠后，越接近设备侧执行计划。

| 图层次 | 主要含义 | 直观理解 | 典型来源 |
| --- | --- | --- | --- |
| Tensor Graph | Tensor 级计算关系 | 数学表达式层 | `x + y`、`matmul`、`sum` |
| Tile Graph | 按 Tile 展开的计算关系 | 分块执行层 | `set_vec_tile_shapes`、`set_cube_tile_shapes` |
| Block Graph | 可划分、可调度的子图结构 | 任务组织层 | 子图划分、依赖分析 |
| Execute Graph | 带依赖和调度信息的执行图 | 设备执行层 | 任务提交与调度 |

Hello World 的 Tensor Graph 很小，只包含输入、加法和输出。复杂算子会让图中出现更多 Operation，例如矩阵乘法、规约、指数、除法、广播和数据搬运。图变大以后，依赖关系和执行组织就会变得更重要。


## 5. API 到计算图的对应关系

可以用下面的表格把 API 和计算图层次对应起来：

| API 或写法 | 主要影响的层次 | 说明 |
| --- | --- | --- |
| `pypto.Tensor([...], dtype)` | Tensor Graph 入口 | 声明图的输入输出参数 |
| `x + y`、`pypto.add(x, y)` | Tensor Graph | 生成逐元素加法 Operation |
| `pypto.matmul(a, b, out_dtype=...)` | Tensor Graph | 生成矩阵乘法 Operation |
| `pypto.sum(x, dim=..., keepdim=...)` | Tensor Graph | 生成规约 Operation |
| `pypto.set_vec_tile_shapes(...)` | Tile Graph | 描述向量类计算的 Tile 组织 |
| `pypto.set_cube_tile_shapes(...)` | Tile Graph | 描述矩阵类计算的 Tile 组织 |
| `pypto.loop(...)`、`pypto.cond(...)` | Tensor Graph / Block Graph | 描述动态循环或条件结构 |
| `out[:] = ...`、`out.move(...)` | Tensor Graph 输出 | 连接计算结果和输出 Tensor |

这个对应关系说明：API 不是孤立调用，而是在构造图。参数描述给出图的边界，Operation 给出图的节点，Tile API 给出执行组织线索，输出写回确定结果落点。


## 6. 从计算图到 MPMD 的关系

MPMD 在本节中只作为计算图执行阶段的视角出现。可以把完整链路概括为：

```text
API 描述
  -> Tensor Graph
  -> Tile Graph
  -> Block Graph
  -> Execute Graph
  -> MPMD 调度执行
```

对于 Hello World，加法 Operation 很少，MPMD 的复杂性并不明显。对于 Softmax、LayerNorm、Attention 片段，同一张图中会同时出现规约、逐元素、矩阵计算和数据搬运。不同任务需要不同程序片段配合，MPMD 执行模型的作用就会变得更清楚。


## 7. 常见混淆点

| 容易混淆的点 | 正确理解 |
| --- | --- |
| `pypto.Tensor(...)` 是否创建真实数据 | 它描述 kernel 参数，不负责创建真实 Tensor 数据 |
| `torch.rand(...)` 是否进入 PyPTO 计算图 | 它属于 Host 侧数据构造，不进入 PyPTO 计算图 |
| Tile Shape 是否改变数学结果 | 不改变数学语义，只影响设备侧分块组织方式 |
| `out[:] = ...` 是否只是 Python 赋值 | 在 JIT kernel 中，它描述输出写回 |
| `npu:0` 是否一定是物理 0 卡 | 它表示当前进程可见设备中的第 0 个逻辑设备 |
| `SIM` 是否等同于真实 NPU 性能 | SIM 用于理解流程和功能验证，不代表真实硬件性能 |


## 8. 本章闭环

本章可以收束为下面这张表：

| 知识点 | 对应能力 |
| --- | --- |
| Hello World 加法算子 | 看到一个完整 PyPTO kernel 的定义、调用和验证 |
| Host/Kernel 划分 | 判断一行代码是在准备数据，还是在描述设备侧计算 |
| PyPTO API 分类 | 知道 JIT、Tensor、dtype、Tile、Operation、控制流分别解决什么问题 |
| 计算图层次 | 理解 Tensor 表达如何逐步变成执行图 |
| MPMD 执行模型 | 理解不同类型任务如何按依赖关系组织执行 |

后续进入初级计算算子章节时，每个新算子都可以放回这条主线中理解：先看数学目标，再看输入输出规格，然后看 kernel 中的 Tensor 表达、Tile 设置和输出写回，最后用 PyTorch reference 建立验证闭环。


## 9. 概念练习

下面的表格给出 Hello World 加法算子中常见语句的 API 与图层次分类。

| 语句或概念 | API 类别 | 图或执行层次 |
| --- | --- | --- |
| `@pypto.frontend.jit(...)` | JIT 与运行配置 | Kernel 入口 |
| `pypto.Tensor([...], pypto.DT_FP32)` | Tensor 参数描述 | Tensor Graph 边界 |
| `pypto.set_vec_tile_shapes(1, 4, 1, 64)` | Tile 配置 | Tile Graph |
| `out[:] = x + y` | Tensor Operation 与输出写回 | Tensor Graph |
| `torch.add(input0, input1)` | Host 侧参考实现 | 不进入 PyPTO 计算图 |
| `assert_allclose(...)` | Host 侧验证 | 不进入 PyPTO 计算图 |

这个练习的重点不是记住单个示例，而是形成稳定的判断方式：先看一行代码属于哪类 API，再看它是否参与 PyPTO 计算图，以及它主要影响哪一层图结构。
